In [0]:
from pyspark.sql.functions import col,to_timestamp,coalesce, lit, regexp_replace

In [0]:
bronze_df = spark.table(
    "workspace.retail_pipeline_test.bronze_sales_v2"
)

bronze_df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_system: string (nullable = true)



In [0]:
silver_df=(bronze_df
            .withColumn("Quantity",col("Quantity").cast("int"))
            .withColumn("UnitPrice",col("UnitPrice").cast("decimal(10,2)"))
            .withColumn("InvoiceDate",to_timestamp(col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss"))
            .withColumn("CustomerID",regexp_replace(coalesce(col("CustomerID"), lit("-1")),r"\.0$",""))
            .withColumn("Description",coalesce(col("Description"),lit("Unknown Product")))
            
            
            
            
            
            
            
            
            
            
            
            
)
            

silver_df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = false)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- CustomerID: string (nullable = false)
 |-- Country: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_system: string (nullable = true)



In [0]:
silver_df.filter(
    col("Description").isNull()
).count()

0

In [0]:
silver_df.groupBy("InvoiceNo", "StockCode").count().show(10)

+---------+---------+-----+
|InvoiceNo|StockCode|count|
+---------+---------+-----+
|   536365|   85123A|    1|
|   536365|    71053|    1|
|   536365|   84406B|    1|
|   536365|   84029G|    1|
|   536365|   84029E|    1|
|   536365|    22752|    1|
|   536365|    21730|    1|
|   536366|    22633|    1|
|   536366|    22632|    1|
|   536367|    84879|    1|
+---------+---------+-----+
only showing top 10 rows


In [0]:
duplicate_groups = (
    silver_df
    .groupBy("InvoiceNo", "StockCode")
    .count()
    .filter(col("count") > 1)
)

duplicate_groups.show(10)

+---------+---------+-----+
|InvoiceNo|StockCode|count|
+---------+---------+-----+
|   536381|    71270|    2|
|   536409|   90199C|    3|
|   536409|    22111|    2|
|   536409|    22866|    2|
|   536409|    85116|    2|
|   536409|    21866|    2|
|   536409|    22900|    2|
|   536412|    22327|    2|
|   536412|    22273|    3|
|   536412|    22940|    2|
+---------+---------+-----+
only showing top 10 rows


In [0]:
transaction_columns = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country"
]

exact_duplicate_groups = (
    silver_df
    .groupBy(*transaction_columns)
    .count()
    .filter(col("count") > 1)
)

exact_duplicate_groups.show(10, truncate=False)

+---------+---------+---------------------------------+--------+-------------------+---------+----------+--------------+-----+
|InvoiceNo|StockCode|Description                      |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |count|
+---------+---------+---------------------------------+--------+-------------------+---------+----------+--------------+-----+
|536409   |22111    |SCOTTIE DOG HOT WATER BOTTLE     |1       |2010-12-01 11:45:00|4.95     |17908.0   |United Kingdom|2    |
|536409   |22866    |HAND WARMER SCOTTY DOG DESIGN    |1       |2010-12-01 11:45:00|2.10     |17908.0   |United Kingdom|2    |
|536409   |21866    |UNION JACK FLAG LUGGAGE TAG      |1       |2010-12-01 11:45:00|1.25     |17908.0   |United Kingdom|2    |
|536409   |22900    | SET 2 TEA TOWELS I LOVE LONDON  |1       |2010-12-01 11:45:00|2.95     |17908.0   |United Kingdom|2    |
|536412   |22327    |ROUND SNACK BOXES SET OF 4 SKULLS|1       |2010-12-01 11:49:00|2.95     |17920.0   |United

In [0]:
exact_duplicate_groups.count()

4879

In [0]:
print("Bronze:", bronze_df.count())
print("Silver:", silver_df.count())

Bronze: 541909
Silver: 541909


In [0]:
silver_df.filter(
    col("Quantity").isNull()
).count()

0

In [0]:
silver_df.filter(
    col("UnitPrice") == 0
).select(
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "UnitPrice",
    "InvoiceDate"
).show(truncate=False)

+---------+---------+---------------+--------+---------+-------------------+
|InvoiceNo|StockCode|Description    |Quantity|UnitPrice|InvoiceDate        |
+---------+---------+---------------+--------+---------+-------------------+
|536414   |22139    |Unknown Product|56      |0.00     |2010-12-01 11:52:00|
|536545   |21134    |Unknown Product|1       |0.00     |2010-12-01 14:32:00|
|536546   |22145    |Unknown Product|1       |0.00     |2010-12-01 14:33:00|
|536547   |37509    |Unknown Product|1       |0.00     |2010-12-01 14:33:00|
|536549   |85226A   |Unknown Product|1       |0.00     |2010-12-01 14:34:00|
|536550   |85044    |Unknown Product|1       |0.00     |2010-12-01 14:34:00|
|536552   |20950    |Unknown Product|1       |0.00     |2010-12-01 14:34:00|
|536553   |37461    |Unknown Product|3       |0.00     |2010-12-01 14:35:00|
|536554   |84670    |Unknown Product|23      |0.00     |2010-12-01 14:35:00|
|536589   |21777    |Unknown Product|-10     |0.00     |2010-12-01 16:50:00|

In [0]:
silver_df.filter(
    col("UnitPrice") == 0
).count()

2519

In [0]:
silver_df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = false)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- CustomerID: string (nullable = false)
 |-- Country: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_system: string (nullable = true)



In [0]:
spark.table(
    "workspace.retail_pipeline_test.silver_sales_v2"
).count()

541909

In [0]:
silver_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.retail_pipeline_test.silver_sales_v2")

In [0]:
silver_table_count = spark.table(
    "workspace.retail_pipeline_test.silver_sales_v2"
).count()

print("Saved Silver Rows:", silver_table_count)

Saved Silver Rows: 541909


In [0]:
from pyspark.sql.functions import col

saved_silver = spark.table(
    "workspace.retail_pipeline_test.silver_sales_v2"
)

saved_silver.select(
    "InvoiceNo",
    "Quantity",
    "UnitPrice",
    "InvoiceDate",
    "CustomerID",
    "Description"
).show(5, truncate=False)

+---------+--------+---------+-------------------+----------+-----------------------------------+
|InvoiceNo|Quantity|UnitPrice|InvoiceDate        |CustomerID|Description                        |
+---------+--------+---------+-------------------+----------+-----------------------------------+
|536365   |6       |2.55     |2010-12-01 08:26:00|17850.0   |WHITE HANGING HEART T-LIGHT HOLDER |
|536365   |6       |3.39     |2010-12-01 08:26:00|17850.0   |WHITE METAL LANTERN                |
|536365   |8       |2.75     |2010-12-01 08:26:00|17850.0   |CREAM CUPID HEARTS COAT HANGER     |
|536365   |6       |3.39     |2010-12-01 08:26:00|17850.0   |KNITTED UNION FLAG HOT WATER BOTTLE|
|536365   |6       |3.39     |2010-12-01 08:26:00|17850.0   |RED WOOLLY HOTTIE WHITE HEART.     |
+---------+--------+---------+-------------------+----------+-----------------------------------+
only showing top 5 rows


In [0]:
saved_silver.filter(
    col("CustomerID").endswith(".0")
).count()

0

In [0]:
spark.sql("""
UPDATE workspace.retail_pipeline_test.silver_sales_v2
SET CustomerID = regexp_replace(CustomerID, '\\\\.0$', '')
WHERE CustomerID RLIKE '\\\\.0$'
""")

DataFrame[num_affected_rows: bigint]

In [0]:
spark.table(
    "workspace.retail_pipeline_test.silver_sales_v2"
).count()

541909